# Student Performance Data Analysis

Exploratory data analysis using Python (Pandas, NumPy, Matplotlib, Seaborn).

Goals:
- Explore correlations between study habits and grades
- Visualize dataset patterns
- Build a simple predictive model

## Dataset
We use the **Student Performance** dataset from the **UCI Machine Learning Repository**.


In [ ]:
# If needed, uncomment the next line the first time you run the notebook
# !pip install ucimlrepo pandas numpy matplotlib scikit-learn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.linear_model import LinearRegression


## 1. Load the dataset

In [ ]:
student_performance = fetch_ucirepo(id=320)

X = student_performance.data.features.copy()
y = student_performance.data.targets.copy()

df = pd.concat([X, y], axis=1)

print("Shape:", df.shape)
display(df.head())


## 2. Quick overview

In [ ]:
print("Column types:")
display(df.dtypes)

print("\nMissing values per column:")
display(df.isna().sum().sort_values(ascending=False).head(10))

print("\nNumeric summary:")
display(df.describe())


## 3. Focus on useful variables

The final grade is **G3**.  
For a first analysis, we focus on variables that are plausibly linked to academic outcomes:

| Variable | Type | Description |
|----------|------|-------------|
| `studytime` | ordinal (1–4) | Weekly study time |
| `failures` | int | Number of past class failures |
| `absences` | int | Number of school absences |
| `internet` | binary | Internet access at home |
| `higher` | binary | Wants to pursue higher education |
| `G1`, `G2` | int | First and second period grades |
| `G3` | int | **Target** — final period grade |

In [ ]:
selected_cols = ["studytime", "failures", "absences", "internet", "higher", "G1", "G2", "G3"]
display(df[selected_cols].head())


## 4. Correlation analysis

In [ ]:
numeric_df = df.select_dtypes(include=[np.number])
corr = numeric_df.corr(numeric_only=True)

corr_target = corr["G3"].sort_values(ascending=False)
display(corr_target)

plt.figure(figsize=(8, 6))
plt.imshow(corr, aspect="auto")
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.columns)), corr.columns)
plt.title("Correlation matrix (numeric variables)")
plt.colorbar()
plt.tight_layout()
plt.show()


## 5. Visualizations

In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(df["G3"], bins=15)
plt.title("Distribution of final grade (G3)")
plt.xlabel("Final grade")
plt.ylabel("Number of students")
plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 4))
plt.scatter(df["studytime"], df["G3"])
plt.title("Study time vs final grade")
plt.xlabel("Study time")
plt.ylabel("G3")
plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 4))
plt.scatter(df["absences"], df["G3"])
plt.title("Absences vs final grade")
plt.xlabel("Absences")
plt.ylabel("G3")
plt.tight_layout()
plt.show()

mean_by_failures = df.groupby("failures")["G3"].mean()
plt.figure(figsize=(6, 4))
plt.bar(mean_by_failures.index.astype(str), mean_by_failures.values)
plt.title("Average final grade by number of past failures")
plt.xlabel("Failures")
plt.ylabel("Average G3")
plt.tight_layout()
plt.show()


## 6. Simple interpretation

Typical observations you can comment on in your README or during an interview:
- `G1` and `G2` are usually strongly related to `G3`
- more past `failures` often correspond to lower final grades
- `studytime` can show a positive tendency, even if it is not perfectly linear
- `absences` may negatively affect performance


## 7. Small predictive baseline

This is **not** meant to be a full ML project.  
The goal is only to show that you can:
- prepare features
- separate numeric and categorical columns
- build a preprocessing pipeline
- train a simple regression model
- evaluate the result


In [ ]:
target = "G3"
features = [c for c in df.columns if c != target]

X = df[features]
y = df[target]

numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(exclude=[np.number]).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model.fit(X_train, y_train)
pred = model.predict(X_test)

print("MAE:", round(mean_absolute_error(y_test, pred), 3))
print("R² :", round(r2_score(y_test, pred), 3))
